Comenzamos la evaluación del finetuning mediante Ragas cargando las librerías pertinentes, que nos dieron bastantes problemas, solucionados finalmente por Claude, porque Gemini se vio superado.

In [ ]:
!pip install --upgrade pip -q
!pip install ragas langchain-community langchain-openai pandas datasets langchain-google-vertexai -q

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 1.8/1.8 MB 8.0 MB/s eta 0:00:00
ERROR: pip's dependency resolver does not currently take into account all the packages that are installed. This behaviour is the source of the following dependency conflicts.
google-colab 1.0.0 requires requests==2.32.4, but you have requests 2.34.2 which is incompatible.


In [ ]:
# 1. Instalación de Unsloth ligero y dependencias de inferencia
!pip install --no-deps "unsloth[colab-new] @ git+https://github.com/unslothai/unsloth.git"
!pip install --no-deps xformers peft accelerate bitsandbytes

  Cloning https://github.com/unslothai/unsloth.git to /tmp/pip-install-7mbw3hfk/unsloth_f749d26e32724622b39bc16aa64c2fd3
  Running command git clone --filter=blob:none --quiet https://github.com/unslothai/unsloth.git /tmp/pip-install-7mbw3hfk/unsloth_f749d26e32724622b39bc16aa64c2fd3
  Resolved https://github.com/unslothai/unsloth.git to commit 37d3b6fdf7d3b18684a88f13c463a6db22e6e2ce
  Installing build dependencies ... done
  Getting requirements to build wheel ... done
  Preparing metadata (pyproject.toml) ... done
  Created wheel for unsloth: filename=unsloth-2026.8.22-py3-none-any.whl size=45113769 sha256=af07a5219e1b75f5c21d94f775a1b15803e6b1becf47bf160da9d6e3f194bb9a
  Stored in directory: /tmp/pip-ephem-wheel-cache-v17r_ce_/wheels/d5/36/1d/4e65996c5b80c84a5ac1b0ba10718bdc155f8dd04352746a8f
Successfully built unsloth
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 3.3/3.3 MB 51.2 MB/s  0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 43.1/43.1 MB 58.8 MB/s  0:00:00
   ━━━━━━━━━━━━━

In [ ]:
pip install unsloth_zoo

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 1.5/1.5 MB 18.8 MB/s  0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 10.2/10.2 MB 60.2 MB/s  0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 3.4/3.4 MB 120.9 MB/s  0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 3.6/3.6 MB 92.0 MB/s  0:00:00
  Attempting uninstall: torchao
    Found existing installation: torchao 0.10.0
    Uninstalling torchao-0.10.0:
      Successfully uninstalled torchao-0.10.0
  Attempting uninstall: datasets
    Found existing installation: datasets 4.0.0
    Uninstalling datasets-4.0.0:
      Successfully uninstalled datasets-4.0.0
  Attempting uninstall: transformers
    Found existing installation: transformers 5.15.1
    Uninstalling transformers-5.15.1:
      Successfully uninstalled transformers-5.15.1
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 9/9 [unsloth_zoo]


La librería más problemática fue llama-cpp-python que cargamos inicialmente de forma simple, como recomienda HF, pero que Claude rectificó, con el código de la celda siguiente.

In [ ]:
!pip install -U llama-cpp-python

  Using cached llama_cpp_python-0.3.35-py3-none-linux_x86_64.whl


In [ ]:
# Cell de instalación: sustituye la celda 5 por esto
!pip uninstall -y llama-cpp-python

# Wheel precompilada para CUDA 12.x (Colab T4)
!pip install llama-cpp-python \
  --extra-index-url https://abetlen.github.io/llama-cpp-python/whl/cu122 \
  --no-cache-dir -q

Found existing installation: llama_cpp_python 0.3.35
Uninstalling llama_cpp_python-0.3.35:
  Successfully uninstalled llama_cpp_python-0.3.35


Tuvimos problemas porque el modelo no se cargaba en la GPU, sino en la RAM del sistema, lo que dificultaba la inferencia del modelo Gemma 3. La celda siguiente mostraba que la GPU estaba disponible.

In [ ]:
# Celda de diagnóstico - ejecutar ANTES de cargar el modelo
import subprocess
result = subprocess.run(['nvidia-smi'], capture_output=True, text=True)
print(result.stdout)

from llama_cpp import Llama
import llama_cpp
print(f"llama-cpp-python version: {llama_cpp.__version__}")

# Verificar que la build tiene soporte CUDA
print(f"CUDA disponible según llama_cpp: {llama_cpp.llama_supports_gpu_offload()}")

Fri Aug 28 08:40:15 2026       
+-----------------------------------------------------------------------------------------+
| NVIDIA-SMI 580.82.07              Driver Version: 580.82.07      CUDA Version: 13.0     |
+-----------------------------------------+------------------------+----------------------+
| GPU  Name                 Persistence-M | Bus-Id          Disp.A | Volatile Uncorr. ECC |
| Fan  Temp   Perf          Pwr:Usage/Cap |           Memory-Usage | GPU-Util  Compute M. |
|                                         |                        |               MIG M. |
|=========================================+========================+======================|
|   0  Tesla T4                       Off |   00000000:00:04.0 Off |                    0 |
| N/A   47C    P8             12W /   70W |       0MiB /  15360MiB |      0%      Default |
|                                         |                        |                  N/A |
+-----------------------------------------+-----

ggml_cuda_init: found 1 CUDA devices (Total VRAM: 14912 MiB):
  Device 0: Tesla T4, compute capability 7.5, VMM: yes, VRAM: 14912 MiB


llama-cpp-python version: 0.3.35
CUDA disponible según llama_cpp: True


Cargamos las librerías de OpenAI, y especialmente las métricas de Ragas, que utilizaremos después.

In [ ]:
import sys
import types

# 1. Parche en memoria para evitar el conflicto con langchain_community / VertexAI
dummy_module = types.ModuleType("langchain_community.chat_models.vertexai")
dummy_module.ChatVertexAI = type("ChatVertexAI", (object,), {})
sys.modules["langchain_community.chat_models.vertexai"] = dummy_module

import os
import pandas as pd
from datasets import Dataset

# Librerías de LangChain
from langchain_openai import ChatOpenAI
from langchain_community.embeddings import HuggingFaceEmbeddings

# 2. Imports actualizados de Ragas (usando 'ragas.metrics.collections')
from ragas import evaluate
from ragas.metrics.collections import (
    faithfulness,
    answer_relevancy,
    context_precision,
    context_recall,
)


print("✅ Todos los módulos e imports se han cargado sin ningún warning.")

/tmp/ipykernel_48715/1819823276.py:15: DeprecationWarning: `langchain-community` is being sunset and is no longer actively maintained. See https://github.com/langchain-ai/langchain-community/issues/674 for details and migration guidance toward standalone integration packages.
  from langchain_community.embeddings import HuggingFaceEmbeddings


✅ Todos los módulos e imports se han cargado sin ningún warning.


Cargamos preliminarmente la clave de DeepSeek, en la celda siguiente codificamos el judge. Repetiremos el proceso más adelante, dentro de la celda donde ejecutamos Ragas.

In [ ]:
from google.colab import userdata
DEEPSEEK_API_KEY = userdata.get('DEEPSEEK_API_KEY')

In [ ]:
# os.environ["OPENAI_API_KEY"] = "TU_API_KEY_AQUI"
DEEPSEEK_BASE_URL = "https://openrouter.ai/api/v1" # Ajusta si usas otro proveedor

# 2. Inicializar el Judge con DeepSeek V4 Pro
deepseek_judge = ChatOpenAI(
    model="deepseek/deepseek-v4-pro", # Ajusta el identificador según tu proveedor (ej. deepseek-ai/DeepSeek-V4-Pro)
    openai_api_key=DEEPSEEK_API_KEY,
    openai_api_base=DEEPSEEK_BASE_URL,
    temperature=0.0 # Se recomienda 0 para evaluación consistente
)

# 3. Embeddings (Ragas usa embeddings para calcular métricas como answer_relevance)
# Puedes usar un modelo de HuggingFace ligero o el de OpenAI/proveedor
from langchain_community.embeddings import HuggingFaceEmbeddings
eval_embeddings = HuggingFaceEmbeddings(model_name="sentence-transformers/paraphrase-multilingual-MiniLM-L12-v2")


/tmp/ipykernel_48715/3952755969.py:15: LangChainDeprecationWarning: The class `HuggingFaceEmbeddings` was deprecated in LangChain 0.2.2 and will be removed in 1.0. An updated version of the class exists in the `langchain-huggingface package and should be used instead. To use it run `pip install -U `langchain-huggingface` and import as `from `langchain_huggingface import HuggingFaceEmbeddings``.
  eval_embeddings = HuggingFaceEmbeddings(model_name="sentence-transformers/paraphrase-multilingual-MiniLM-L12-v2")


Loading weights:   0%|          | 0/199 [00:00<?, ?it/s]

BertModel LOAD REPORT from: sentence-transformers/paraphrase-multilingual-MiniLM-L12-v2
Key                     | Status     |  | 
------------------------+------------+--+-
embeddings.position_ids | UNEXPECTED |  | 

Notes:
- UNEXPECTED:	can be ignored when loading from different task/architecture; not ok if you expect identical arch.


Ahora tenemos que configurar el diccionario que pasaremos a Ragas, cargando el fichero de test con los ejemplos en formato .jsonl. Pasamos estos a un dataframe de Pandas, y de ahí al diccionario con el formato Ragas, a falta de las respuestas del modelo Gemma3.

In [ ]:
import pandas as pd
from datasets import Dataset

In [ ]:
# Carga de los ejemplos de text, con lo que probaremos el modelo
import pandas as pd
df_eval = pd.read_json('/content/drive/MyDrive/Ollama_app_comentario/test/dataset_market_commentary_test.jsonl', lines=True) # OJO!!

In [ ]:
# Preparación de los datos para introducirlos en el diccionario requerido por Ragas

df_eval['instructions'] = df_eval['messages'].apply(lambda x: x[0]['content'])
df_eval['contexts'] = df_eval['messages'].apply(lambda x: x[1]['content'])
df_eval['ground_truth'] = df_eval['messages'].apply(lambda x: x[2]['content'])
df_eval['question'] = df_eval['instructions'] + '\n\n' + df_eval['contexts']

question = df_eval['question'].tolist()
ground_truth = df_eval['ground_truth'].tolist()
contexts = [[item] for item in df_eval['contexts'].tolist()]

In [ ]:
# Relleno de campos necesarios del diccionario data, requerido por Ragas

data = {}
data['question'] = question
data['contexts'] = contexts
data['ground_truth'] = ground_truth

Cargamos torch y librerías de unsloth para poder descargar el modelo Gemma 3 en local.

In [ ]:
from unsloth import FastLanguageModel
import torch

/usr/local/lib/python3.13/dist-packages/unsloth/__init__.py:1531: UserWarning: WARNING: Unsloth should be imported before [transformers, peft] to ensure all optimizations are applied. Your code may run slower or encounter memory issues without these optimizations.

Please restructure your imports with 'import unsloth' at the top of your file.
  from ._gpu_init import *


🦥 Unsloth: Will patch your computer to enable 2x faster free finetuning.
🦥 Unsloth Zoo will now patch everything to make training faster!


Descargamos el modelo de Gemma3 desde Hugging Face.

In [ ]:
from huggingface_hub import hf_hub_download
from llama_cpp import Llama

# 1. Descargar únicamente el archivo .gguf
model_path = hf_hub_download(
    repo_id="GuillermoBarrio/gemma3-4b-finetuned-comentarios-gguf",
    filename="gemma-3-4b-it.Q4_K_M.gguf"
)

# 2. Cargar modelo en GPU

model = Llama(
    model_path=model_path,
    n_ctx=6000,
    n_gpu_layers=99,
    n_batch=512,
    flash_attn=True,
    verbose=True   # ← cámbialo a True temporalmente
)


print("✅ Modelo cargado con éxito en VRAM")

llama_model_loader: loaded meta data with 39 key-value pairs and 444 tensors from /root/.cache/huggingface/hub/models--GuillermoBarrio--gemma3-4b-finetuned-comentarios-gguf/snapshots/3ef2f2ff5b5311b762bea45fddbe7f0eaf4095bf/gemma-3-4b-it.Q4_K_M.gguf (version GGUF V3 (latest))
llama_model_loader: Dumping metadata keys/values. Note: KV overrides do not apply in this output.
llama_model_loader: - kv   0:                       general.architecture str              = gemma3
llama_model_loader: - kv   1:                               general.type str              = model
llama_model_loader: - kv   2:                     general.sampling.top_k i32              = 64
llama_model_loader: - kv   3:                     general.sampling.top_p f32              = 0.950000
llama_model_loader: - kv   4:                               general.name str              = Unsloth_Gguf_Giis7Gso
llama_model_loader: - kv   5:                       general.quantized_by str              = Unsloth
llama_model_loader

✅ Modelo cargado con éxito en VRAM


Ahora inferimos el modelo de Gemma 3 para obtener los comentarios en base a los 7 ejemplos de prompt del conjunto de test. Almecelamos las respuestas en el campo 'answers' del diccionario requerido por Ragas.

In [ ]:
# 3. Generar las respuestas con tu modelo Fine-Tuned (usando llama-cpp-python)
answers = []
print("Generando respuestas con Gemma 3 Fine-Tuned para el test...")

n = 0

for system_instruction, user_data in zip(df_eval['instructions'], df_eval['contexts']):
    # Formato de mensajes para el motor de chat
    n += 1

    print(n)

    messages = [
        {"role": "system", "content": system_instruction},
        {"role": "user", "content": user_data}
    ]

    # Inferencia directa usando la API de chat de llama-cpp-python
    response = model.create_chat_completion(
        messages=messages,
        max_tokens=2000,
        temperature=0.1
    )

    # Extraer el texto generado por la respuesta del asistente
    respuesta = response["choices"][0]["message"]["content"]
    answers.append(respuesta)

print("✅ Todas las respuestas generadas con éxito.")

# 4. Estructurar el diccionario 'data' para Ragas
data = {
    "question": df_eval['question'].tolist(),
    "contexts": [[item] for item in df_eval['contexts'].tolist()], # Solo los datos brutos como fuente de verdad
    "answer": answers,                                             # Salidas generadas por Gemma 3
    "ground_truth": df_eval['ground_truth'].tolist()               # Comentario humano ideal
}

# 5. Convertir a Dataset de Hugging Face
eval_dataset = Dataset.from_dict(data)
print("✅ 'data' y 'eval_dataset' construidos correctamente con las 4 columnas requeridas.")

Generando respuestas con Gemma 3 Fine-Tuned para el test...
1


CUDA Graph id 87 reused
ggml_backend_cuda_graph_compute: CUDA graph warmup complete
CUDA Graph id 87 reused
CUDA Graph id 87 reused
CUDA Graph id 87 reused
CUDA Graph id 87 reused
CUDA Graph id 87 reused
CUDA Graph id 87 reused
CUDA Graph id 87 reused
CUDA Graph id 87 reused
CUDA Graph id 87 reused
CUDA Graph id 87 reused
CUDA Graph id 87 reused
CUDA Graph id 87 reused
CUDA Graph id 87 reused
CUDA Graph id 87 reused
CUDA Graph id 87 reused
CUDA Graph id 87 reused
CUDA Graph id 87 reused
CUDA Graph id 87 reused
CUDA Graph id 87 reused
CUDA Graph id 87 reused
CUDA Graph id 87 reused
CUDA Graph id 87 reused
CUDA Graph id 87 reused
CUDA Graph id 87 reused
CUDA Graph id 87 reused
CUDA Graph id 87 reused
CUDA Graph id 87 reused
CUDA Graph id 87 reused
CUDA Graph id 87 reused
CUDA Graph id 87 reused
CUDA Graph id 87 reused
CUDA Graph id 87 reused
CUDA Graph id 87 reused
CUDA Graph id 87 reused
CUDA Graph id 87 reused
CUDA Graph id 87 reused
CUDA Graph id 87 reused
CUDA Graph id 87 reused
CUDA

2


CUDA Graph id 132 reused
ggml_backend_cuda_graph_compute: CUDA graph warmup complete
CUDA Graph id 132 reused
CUDA Graph id 132 reused
CUDA Graph id 132 reused
CUDA Graph id 132 reused
CUDA Graph id 132 reused
CUDA Graph id 132 reused
CUDA Graph id 132 reused
CUDA Graph id 132 reused
CUDA Graph id 132 reused
CUDA Graph id 132 reused
CUDA Graph id 132 reused
CUDA Graph id 132 reused
CUDA Graph id 132 reused
CUDA Graph id 132 reused
CUDA Graph id 132 reused
CUDA Graph id 132 reused
CUDA Graph id 132 reused
CUDA Graph id 132 reused
CUDA Graph id 132 reused
CUDA Graph id 132 reused
CUDA Graph id 132 reused
CUDA Graph id 132 reused
CUDA Graph id 132 reused
CUDA Graph id 132 reused
CUDA Graph id 132 reused
CUDA Graph id 132 reused
CUDA Graph id 132 reused
CUDA Graph id 132 reused
CUDA Graph id 132 reused
CUDA Graph id 132 reused
CUDA Graph id 132 reused
CUDA Graph id 132 reused
CUDA Graph id 132 reused
CUDA Graph id 132 reused
CUDA Graph id 132 reused
CUDA Graph id 132 reused
CUDA Graph id 1

3


...

6


CUDA Graph id 306 reused
ggml_backend_cuda_graph_compute: CUDA graph warmup complete
CUDA Graph id 306 reused
CUDA Graph id 306 reused
CUDA Graph id 306 reused
CUDA Graph id 306 reused
CUDA Graph id 306 reused
CUDA Graph id 306 reused
CUDA Graph id 306 reused
CUDA Graph id 306 reused
CUDA Graph id 306 reused
CUDA Graph id 306 reused
CUDA Graph id 306 reused
CUDA Graph id 306 reused
CUDA Graph id 306 reused
CUDA Graph id 306 reused
CUDA Graph id 306 reused
CUDA Graph id 306 reused
CUDA Graph id 306 reused
CUDA Graph id 306 reused
CUDA Graph id 306 reused
CUDA Graph id 306 reused
CUDA Graph id 306 reused
CUDA Graph id 306 reused
CUDA Graph id 306 reused
CUDA Graph id 306 reused
CUDA Graph id 306 reused
CUDA Graph id 306 reused
CUDA Graph id 306 reused
CUDA Graph id 306 reused
CUDA Graph id 306 reused
CUDA Graph id 306 reused
CUDA Graph id 306 reused
CUDA Graph id 306 reused
CUDA Graph id 306 reused
CUDA Graph id 306 reused
CUDA Graph id 306 reused
CUDA Graph id 306 reused
CUDA Graph id 3

7


CUDA Graph id 345 reused
ggml_backend_cuda_graph_compute: CUDA graph warmup complete
CUDA Graph id 345 reused
CUDA Graph id 345 reused
CUDA Graph id 345 reused
CUDA Graph id 345 reused
CUDA Graph id 345 reused
CUDA Graph id 345 reused
CUDA Graph id 345 reused
CUDA Graph id 345 reused
CUDA Graph id 345 reused
CUDA Graph id 345 reused
CUDA Graph id 345 reused
CUDA Graph id 345 reused
CUDA Graph id 345 reused
CUDA Graph id 345 reused
CUDA Graph id 345 reused
CUDA Graph id 345 reused
CUDA Graph id 345 reused
CUDA Graph id 345 reused
CUDA Graph id 345 reused
CUDA Graph id 345 reused
CUDA Graph id 345 reused
CUDA Graph id 345 reused
CUDA Graph id 345 reused
CUDA Graph id 345 reused
CUDA Graph id 345 reused
CUDA Graph id 345 reused
CUDA Graph id 345 reused
CUDA Graph id 345 reused
CUDA Graph id 345 reused
CUDA Graph id 345 reused
CUDA Graph id 345 reused
CUDA Graph id 345 reused
CUDA Graph id 345 reused
CUDA Graph id 345 reused
CUDA Graph id 345 reused
CUDA Graph id 345 reused
CUDA Graph id 3

✅ Todas las respuestas generadas con éxito.
✅ 'data' y 'eval_dataset' construidos correctamente con las 4 columnas requeridas.


Echamos un vistazo a las respuestas del modelo. Se ve que en el último caso el modelo repite de forma sistemática una frase, lo cual no es precisamente bueno.

In [ ]:
answers

['## ANÁLISIS DE MERCADOS - 20 DE AGOSTO DE 2026\n\n**Rendimiento de Renta Variable (PECENTOS 20/08/2026):**\n\n- S&P500: +0.21%\n- S&P Equal Weight (SPW): +1.02%\n- 7 Magníficas (BM7T): +1.23%\n- EuroStoxx50: -0.37%\n- Stoxx600: -0.11%\n- Ibex35: -0.44%\n\n*Análisis:* El S&P500 y las 7 Magníficas se han apreciado ligeramente, impulsados en parte por datos de norteamericano y en menor medida europeo. El S&P Equal Weight ha superado al S&P500 en más de un punto, y las 7 Magníficas han tenido el mejor rendimiento de todos. El EuroStoxx50 y el Stoxx600 han tenido resultados negativos, principalmente por el sector bancario en Europa. El Ibex35 ha cerrado en rojo, con una diferencia de más de un punto por centos entre sus máximos y mínimos de la sesión.\n\n**Rendimiento de Materias Primas:**\n\n- Petróleo Brent: $91.9/barril\n- Petróleo WTI: $86.0/barril\n\n*Análisis:* Los precios del petróleo han subido ligeramente, por encima de los $92 y $86, respectivamente.\n\n**Rendimiento de Renta Fi

Definimos las métricas de Ragas a determinar.

In [ ]:
# Definir métricas principales de Ragas
metrics = [
    faithfulness,         # ¿La respuesta generada se basa solo en el contexto?
    context_precision,    # ¿El contexto recuperado es relevante para la pregunta?
    answer_relevancy,
    context_recall,       # ¿El contexto contiene toda la información de la ground_truth?
]


Tuvimos varios errores 401, con lo que nos aseguramos inicialmente que podemos contactar con DeepSeek.

In [ ]:
from openai import OpenAI

client = OpenAI(
    api_key=DEEPSEEK_API_KEY,
    base_url="https://api.deepseek.com"   # ← correcto para DeepSeek directo
)

response = client.chat.completions.create(
    model="deepseek-chat",                 # ← nombre del modelo en DeepSeek directo
    messages=[{"role": "user", "content": "Di solo: OK"}],
    max_tokens=10
)
print(response.choices[0].message.content)

¡Hola! 😄

¿Listo


Este es el código de la implementación de Ragas, que define el judge, baja los embeddings de un modelo simple, y calcula las métricas, guardando los resultados y los datos de origen en un dataframe de Pandas.

In [ ]:
from openai import OpenAI
from ragas.llms import llm_factory
from langchain_community.embeddings import HuggingFaceEmbeddings
from ragas.embeddings import LangchainEmbeddingsWrapper

openai_client = OpenAI(
    api_key=DEEPSEEK_API_KEY,
    base_url="https://api.deepseek.com"
)

ragas_judge = llm_factory(
    model="deepseek-chat",
    client=openai_client,
    max_tokens=4096
)

lc_embeddings = HuggingFaceEmbeddings(
    model_name="sentence-transformers/paraphrase-multilingual-MiniLM-L12-v2"
)
ragas_embeddings = LangchainEmbeddingsWrapper(lc_embeddings)

for metric in metrics:
    metric.llm = ragas_judge
    if hasattr(metric, "embeddings"):
        metric.embeddings = ragas_embeddings

print("Ejecutando evaluación con DeepSeek Chat...")
result = evaluate(
    dataset=eval_dataset,
    metrics=metrics,
    llm=ragas_judge,
    embeddings=ragas_embeddings
)

print("\n--- RESULTADOS GLOBALES ---")
print(result)

# Convertir a Pandas DataFrame para análisis detallado fila por fila
df_results = result.to_pandas()
df_results.head(7)

Loading weights:   0%|          | 0/199 [00:00<?, ?it/s]

BertModel LOAD REPORT from: sentence-transformers/paraphrase-multilingual-MiniLM-L12-v2
Key                     | Status     |  | 
------------------------+------------+--+-
embeddings.position_ids | UNEXPECTED |  | 

Notes:
- UNEXPECTED:	can be ignored when loading from different task/architecture; not ok if you expect identical arch.
/tmp/ipykernel_48715/3301101576.py:20: DeprecationWarning: LangchainEmbeddingsWrapper is deprecated and will be removed in a future version. Use the modern embedding providers instead: embedding_factory('openai', model='text-embedding-3-small', client=openai_client) or from ragas.embeddings import OpenAIEmbeddings, GoogleEmbeddings, HuggingFaceEmbeddings
  ragas_embeddings = LangchainEmbeddingsWrapper(lc_embeddings)


Ejecutando evaluación con DeepSeek Chat...


Evaluating:   0%|          | 0/28 [00:00<?, ?it/s]

ERROR:ragas.executor:Exception raised in Job[20]: IncompleteOutputException(The output is incomplete due to a max_tokens length limit.)
ERROR:ragas.executor:Exception raised in Job[24]: IncompleteOutputException(The output is incomplete due to a max_tokens length limit.)



--- RESULTADOS GLOBALES ---
{'faithfulness': 0.4715, 'context_precision': 1.0000, 'answer_relevancy': 0.4055, 'context_recall': 0.9311}


,user_input,retrieved_contexts,response,reference,faithfulness,context_precision,answer_relevancy,context_recall
0,Eres un analista financiero senior. Debes reda...,[## DATOS NUMÉRICOS DE MERCADO (USA ESTOS VALO...,## ANÁLISIS DE MERCADOS - 20 DE AGOSTO DE 2026...,**La renta variable estadounidense** cerró aye...,0.581395,1.0,0.430233,0.967742
1,Eres un analista financiero senior. Debes reda...,[## DATOS NUMÉRICOS DE MERCADO (USA ESTOS VALO...,## Análise de Mercados - 21 de agosto de 2026\...,**Wall Street** encadena su quinta sesión cons...,0.227273,1.0,0.476711,1.000000
2,Eres un analista financiero senior. Debes reda...,[## DATOS NUMÉRICOS DE MERCADO (USA ESTOS VALO...,## COMENTO DE MERCADOS - 24 de agosto de 2026\...,**La renta variable europea afronta una sesión...,0.800000,1.0,0.424005,0.923077
3,Eres un analista financiero senior. Debes reda...,[## DATOS NUMÉRICOS DE MERCADO (USA ESTOS VALO...,## Análise de Mercados - 26 de Agosto de 2026\...,**Las bolsas europeas apuntan a una apertura l...,0.387097,1.0,0.485037,0.960000
4,Eres un analista financiero senior. Debes reda...,[## DATOS NUMÉRICOS DE MERCADO (USA ESTOS VALO...,## COMENTO DE MERCADOS - 26 de agosto de 2026\...,**La renta variable estadounidense cerró ayer ...,0.361702,1.0,0.545854,0.703704
5,Eres un analista financiero senior. Debes reda...,[## DATOS NUMÉRICOS DE MERCADO (USA ESTOS VALO...,## Comentario de Mercados - 27 de Septiembre d...,**Nvidia** ha vuelto a ejercer como catalizado...,NaN,1.0,0.236598,0.962963
6,Eres un analista financiero senior. Debes reda...,[## DATOS NUMÉRICOS DE MERCADO (USA ESTOS VALO...,## ANÁLISIS DE MERCADOS - 28 DE AGOSTO DE 2026...,**La renta variable norteamericana cerró ayer ...,NaN,1.0,0.240248,1.000000


Guardamos los resultados completos en nuestro Drive, así como una versión muy abreviada, solo con los datos de las métricas.

In [ ]:
# Guardar en CSV para anexarlo a la memoria de tu proyecto
df_results.to_csv("/content/drive/MyDrive/Ollama_app_comentario/evaluacion_ragas_deepseek.csv", index=False)
print("Evaluación guardada en Google Drive correctamente.")

Evaluación guardada en Google Drive correctamente.


In [ ]:
df_results.columns

Index(['user_input', 'retrieved_contexts', 'response', 'reference',
       'faithfulness', 'context_precision', 'answer_relevancy',
       'context_recall'],
      dtype='object')

In [ ]:
df_solo_metricas = df_results.drop(columns=['user_input', 'retrieved_contexts', 'response', 'reference'])

In [ ]:
df_solo_metricas.head()

,faithfulness,context_precision,answer_relevancy,context_recall
0,0.581395,1.0,0.430233,0.967742
1,0.227273,1.0,0.476711,1.000000
2,0.800000,1.0,0.424005,0.923077
3,0.387097,1.0,0.485037,0.960000
4,0.361702,1.0,0.545854,0.703704


In [ ]:
df_solo_metricas.to_csv("/content/drive/MyDrive/Ollama_app_comentario/evaluacion_ragas_deepseek_metricas.csv", index=False)
print("Evaluación guardada en Google Drive correctamente.")

Evaluación guardada en Google Drive correctamente.
